# P7: Surface SID Entities into PRI Schema
Creates three new PRI tables from TMF SID source schemas:
1. **service_problem** ← `tmf_service.service_problem` (100K rows) — P7-SURFACE-SERVICE-PROBLEM
2. **work_order** ← `tmf_enterprise.work` (100K rows) — P7-SURFACE-WORK-ORDERS  
3. **sla_breach** ← `tmf_customer.sla_breach` (10K rows) — P7-SURFACE-SLA-BREACH

All tables mapped to PRI-relevant columns with FK keys to existing PRI entities.

In [0]:
%sql
CREATE OR REPLACE TABLE cdm_tmforum.provable_resilience_intelligence.service_problem
COMMENT 'Service problems from TMF SID tmf_service.service_problem, surfaced for PRI root-cause tracing. Maps service_id, customer_id, work_id to enable correlation with proactive_action queue and resilience scores.'
AS
SELECT
  CAST(sp.service_problem_id AS STRING) AS service_problem_id,
  CAST(sp.service_id AS STRING) AS service_id,
  CAST(sp.customer_id AS STRING) AS customer_id,
  CAST(sp.work_id AS STRING) AS work_id,
  CAST(sp.alarm_id AS STRING) AS alarm_id,
  sp.severity,
  sp.status,
  sp.priority,
  sp.category,
  sp.title,
  sp.description,
  sp.detection_method,
  sp.impact_level,
  sp.root_cause_code,
  sp.root_cause_description,
  sp.resolution_code,
  sp.resolution_description,
  sp.escalation_level,
  sp.escalation_reason,
  sp.service_type,
  sp.technology_domain,
  sp.network_function_type,
  sp.geographic_site_reference,
  sp.sla_breach_flag,
  sp.is_recurring,
  sp.is_known_error,
  TRY_CAST(sp.affected_customer_count AS INT) AS affected_customer_count,
  CAST(sp.estimated_revenue_impact AS DOUBLE) AS estimated_revenue_impact_usd,
  TRY_CAST(sp.outage_duration_minutes AS INT) AS outage_duration_minutes,
  TRY_CAST(sp.reported_timestamp AS TIMESTAMP) AS reported_at,
  TRY_CAST(sp.acknowledged_timestamp AS TIMESTAMP) AS acknowledged_at,
  TRY_CAST(sp.outage_start_timestamp AS TIMESTAMP) AS outage_start_at,
  TRY_CAST(sp.outage_end_timestamp AS TIMESTAMP) AS outage_end_at,
  TRY_CAST(sp.resolved_timestamp AS TIMESTAMP) AS resolved_at,
  TRY_CAST(sp.closed_timestamp AS TIMESTAMP) AS closed_at,
  TRY_CAST(sp.created_timestamp AS TIMESTAMP) AS created_at,
  sp.source_system
FROM cdm_tmforum.tmf_service.service_problem sp

In [0]:
%sql
CREATE OR REPLACE TABLE cdm_tmforum.provable_resilience_intelligence.work_order
COMMENT 'Work orders from TMF SID tmf_enterprise.work, surfaced for PRI crew dispatch tracking. Links work_id to proactive_action.action_type=dispatch_crew for fulfillment analysis.'
AS
SELECT
  CAST(w.work_id AS STRING) AS work_id,
  CAST(w.service_id AS STRING) AS service_id,
  CAST(w.customer_id AS STRING) AS customer_id,
  CAST(w.customer_problem_id AS STRING) AS customer_problem_id,
  CAST(w.physical_resource_id AS STRING) AS physical_resource_id,
  CAST(w.order_id AS STRING) AS order_id,
  w.status,
  w.priority,
  w.category,
  w.type AS work_type,
  w.title,
  w.description,
  w.dispatch_method,
  w.outcome,
  w.technician_name,
  w.site_address,
  CAST(w.site_latitude AS DOUBLE) AS site_latitude,
  CAST(w.site_longitude AS DOUBLE) AS site_longitude,
  TRY_CAST(w.estimated_duration_minutes AS INT) AS estimated_duration_minutes,
  TRY_CAST(w.actual_duration_minutes AS INT) AS actual_duration_minutes,
  CAST(w.travel_distance_km AS DOUBLE) AS travel_distance_km,
  TRY_CAST(w.travel_time_minutes AS INT) AS travel_time_minutes,
  CAST(w.labor_cost_amount AS DOUBLE) AS labor_cost_usd,
  CAST(w.parts_cost_amount AS DOUBLE) AS parts_cost_usd,
  CAST(w.total_cost_amount AS DOUBLE) AS total_cost_usd,
  w.parts_used_flag,
  w.sla_met_flag,
  w.escalation_flag,
  w.escalation_reason,
  w.actual_start_timestamp AS started_at,
  w.actual_end_timestamp AS ended_at,
  w.sla_target_completion_timestamp AS sla_target_at,
  w.created_timestamp AS created_at,
  w.source_system
FROM cdm_tmforum.tmf_enterprise.work w

In [0]:
%sql
CREATE OR REPLACE TABLE cdm_tmforum.provable_resilience_intelligence.sla_breach
COMMENT 'SLA breaches from TMF SID tmf_customer.sla_breach, surfaced for PRI SLA→resilience_score correlation. Joins sla_breach.service_id to service_inventory for impact analysis.'
AS
SELECT
  CAST(sb.sla_breach_id AS STRING) AS sla_breach_id,
  CAST(sb.service_id AS STRING) AS service_id,
  CAST(sb.customer_id AS STRING) AS customer_id,
  CAST(sb.service_level_agreement_id AS STRING) AS sla_id,
  sb.severity,
  sb.status,
  sb.breach_type,
  sb.breach_category,
  sb.breach_number,
  sb.sla_metric_name,
  CAST(sb.sla_threshold_value AS DOUBLE) AS sla_threshold_value,
  CAST(sb.actual_value AS DOUBLE) AS actual_value,
  CAST(sb.variance_value AS DOUBLE) AS variance_value,
  CAST(sb.variance_percentage AS DOUBLE) AS variance_pct,
  sb.threshold_unit,
  sb.root_cause_category,
  sb.root_cause_description,
  sb.detection_method,
  sb.escalation_level,
  sb.remediation_action,
  sb.preventive_action,
  CAST(sb.breach_duration_minutes AS DOUBLE) AS breach_duration_minutes,
  sb.is_compensable,
  sb.is_excusable,
  sb.compensation_status,
  sb.compensation_type,
  CAST(sb.compensation_amount AS DOUBLE) AS compensation_amount_usd,
  sb.customer_impact_description,
  TRY_CAST(sb.affected_users_count AS INT) AS affected_users_count,
  sb.customer_notification_sent,
  sb.breach_start_timestamp AS breach_start_at,
  sb.breach_end_timestamp AS breach_end_at,
  sb.detection_timestamp AS detected_at,
  sb.resolution_timestamp AS resolved_at,
  sb.escalation_timestamp AS escalated_at,
  sb.created_timestamp AS created_at,
  CAST(NULL AS STRING) AS source_system
FROM cdm_tmforum.tmf_customer.sla_breach sb

In [0]:
SCHEMA = "cdm_tmforum.provable_resilience_intelligence"

# ── service_problem column comments ──
sp_comments = {
    "service_problem_id": "Unique service problem identifier (PK, cast from TMF bigint)",
    "service_id": "FK to service — TMF service_id (bigint as string for cross-schema join)",
    "customer_id": "FK to customer — affected customer",
    "work_id": "FK to work_order — associated remediation work",
    "alarm_id": "FK to resource_alarm — triggering alarm",
    "severity": "Problem severity: critical | major | minor | warning | cleared | indeterminate",
    "status": "Lifecycle status: submitted | acknowledged | in_progress | pending_customer | pending_third_party | resolved | closed | cancelled",
    "priority": "Priority level of this problem",
    "category": "High-level problem category",
    "title": "Short problem title",
    "description": "Free-text problem description",
    "detection_method": "How the problem was detected",
    "impact_level": "Assessed impact level",
    "root_cause_code": "Root cause classification code",
    "root_cause_description": "Free-text root cause description",
    "resolution_code": "Resolution classification code",
    "resolution_description": "Free-text resolution description",
    "escalation_level": "Current escalation level",
    "escalation_reason": "Reason for escalation",
    "service_type": "Type of affected service",
    "technology_domain": "Technology domain: access | transport | core",
    "network_function_type": "Network function type affected",
    "geographic_site_reference": "Reference to geographic site",
    "sla_breach_flag": "Whether this problem triggered an SLA breach",
    "is_recurring": "Whether this is a recurring problem",
    "is_known_error": "Whether this is a known error",
    "affected_customer_count": "Number of customers affected",
    "estimated_revenue_impact_usd": "Estimated revenue impact in USD",
    "outage_duration_minutes": "Duration of outage in minutes",
    "reported_at": "When the problem was reported",
    "acknowledged_at": "When the problem was acknowledged",
    "outage_start_at": "Outage start timestamp",
    "outage_end_at": "Outage end timestamp",
    "resolved_at": "When the problem was resolved",
    "closed_at": "When the problem was closed",
    "created_at": "Record creation timestamp",
    "source_system": "Originating system of record",
}

# ── work_order column comments ──
wo_comments = {
    "work_id": "Unique work order identifier (PK)",
    "service_id": "FK to service — associated service instance",
    "customer_id": "FK to customer — requesting customer",
    "customer_problem_id": "FK to customer problem that triggered this work",
    "physical_resource_id": "FK to physical resource being worked on",
    "order_id": "FK to originating order",
    "status": "Lifecycle status: open | in_progress | completed | cancelled",
    "priority": "Priority level: critical | high | medium | low",
    "category": "Work category classification",
    "work_type": "Type of work (TMF SID classification)",
    "title": "Work order title",
    "description": "Free-text work description",
    "dispatch_method": "How the crew was dispatched",
    "outcome": "Work outcome result",
    "technician_name": "Assigned technician name",
    "site_address": "Work site street address",
    "site_latitude": "Work site WGS84 latitude",
    "site_longitude": "Work site WGS84 longitude",
    "estimated_duration_minutes": "Estimated work duration in minutes",
    "actual_duration_minutes": "Actual work duration in minutes",
    "travel_distance_km": "Travel distance to site in kilometers",
    "travel_time_minutes": "Travel time to site in minutes",
    "labor_cost_usd": "Labor cost in USD",
    "parts_cost_usd": "Parts cost in USD",
    "total_cost_usd": "Total work order cost in USD",
    "parts_used_flag": "Whether spare parts were consumed",
    "sla_met_flag": "Whether SLA target was met",
    "escalation_flag": "Whether this work was escalated",
    "escalation_reason": "Reason for escalation",
    "started_at": "Actual work start timestamp",
    "ended_at": "Actual work end timestamp",
    "sla_target_at": "SLA target completion timestamp",
    "created_at": "Record creation timestamp",
    "source_system": "Originating system of record",
}

# ── sla_breach column comments ──
sb_comments = {
    "sla_breach_id": "Unique SLA breach identifier (PK)",
    "service_id": "FK to service_inventory — breached service",
    "customer_id": "FK to customer — affected customer",
    "sla_id": "FK to service level agreement",
    "severity": "Breach severity level",
    "status": "Breach lifecycle status",
    "breach_type": "Type of SLA breach",
    "breach_category": "Category of SLA breach",
    "breach_number": "Unique breach reference number",
    "sla_metric_name": "Name of the SLA metric breached",
    "sla_threshold_value": "SLA threshold that was breached",
    "actual_value": "Actual measured value at time of breach",
    "variance_value": "Absolute variance from threshold",
    "variance_pct": "Percentage variance from threshold",
    "threshold_unit": "Unit of measurement for threshold",
    "root_cause_category": "Root cause classification",
    "root_cause_description": "Free-text root cause description",
    "detection_method": "How the breach was detected",
    "escalation_level": "Current escalation level",
    "remediation_action": "Remediation action taken",
    "preventive_action": "Preventive action planned",
    "breach_duration_minutes": "Duration of breach in minutes",
    "is_compensable": "Whether breach qualifies for compensation",
    "is_excusable": "Whether breach is excusable (force majeure etc.)",
    "compensation_status": "Status of compensation: pending | approved | paid | denied",
    "compensation_type": "Type of compensation offered",
    "compensation_amount_usd": "Compensation amount in USD",
    "customer_impact_description": "Description of customer impact",
    "affected_users_count": "Number of end users affected",
    "customer_notification_sent": "Whether customer was notified",
    "breach_start_at": "Breach start timestamp",
    "breach_end_at": "Breach end timestamp",
    "detected_at": "Breach detection timestamp",
    "resolved_at": "Breach resolution timestamp",
    "escalated_at": "Breach escalation timestamp",
    "created_at": "Record creation timestamp",
    "source_system": "Originating system of record",
}

# Apply all comments
tables_comments = [
    ("service_problem", sp_comments),
    ("work_order", wo_comments),
    ("sla_breach", sb_comments),
]

total_cols = 0
for table_name, col_comments in tables_comments:
    for col, comment in col_comments.items():
        safe = comment.replace("'", "''")
        spark.sql(f"ALTER TABLE {SCHEMA}.{table_name} ALTER COLUMN {col} COMMENT '{safe}'")
        total_cols += 1
    print(f"  {table_name}: {len(col_comments)} columns documented")

print(f"\n✅ Total: {total_cols} column comments applied across 3 tables")

In [0]:
%sql
-- Validation summary for all three surfaced tables
WITH counts AS (
  SELECT 'service_problem' AS entity, COUNT(*) AS rows, COUNT(DISTINCT service_problem_id) AS pk_distinct
  FROM cdm_tmforum.provable_resilience_intelligence.service_problem
  UNION ALL
  SELECT 'work_order', COUNT(*), COUNT(DISTINCT work_id)
  FROM cdm_tmforum.provable_resilience_intelligence.work_order
  UNION ALL
  SELECT 'sla_breach', COUNT(*), COUNT(DISTINCT sla_breach_id)
  FROM cdm_tmforum.provable_resilience_intelligence.sla_breach
)
SELECT * FROM counts

In [0]:
%sql
-- Check FK overlap with PRI service_inventory (service_id is STRING in PRI, bigint-cast-to-string from TMF)
-- service_problem→service_inventory
SELECT 
  'sp→si' AS fk_path,
  COUNT(*) AS total_sp_rows,
  COUNT(si.service_id) AS matched,
  COUNT(*) - COUNT(si.service_id) AS orphans
FROM cdm_tmforum.provable_resilience_intelligence.service_problem sp
LEFT JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON sp.service_id = si.service_id

UNION ALL

-- work_order→service_inventory
SELECT 
  'wo→si',
  COUNT(*),
  COUNT(si.service_id),
  COUNT(*) - COUNT(si.service_id)
FROM cdm_tmforum.provable_resilience_intelligence.work_order wo
LEFT JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON wo.service_id = si.service_id

UNION ALL

-- sla_breach→service_inventory  
SELECT 
  'sb→si',
  COUNT(*),
  COUNT(si.service_id),
  COUNT(*) - COUNT(si.service_id)
FROM cdm_tmforum.provable_resilience_intelligence.sla_breach sb
LEFT JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON sb.service_id = si.service_id